# Notebook 01 – Build Historical Dataset

Build a labelled feature dataset for All-Star prediction.

**What this notebook does:**
1. Fetches per-player game logs from the NBA API for each training season.
2. Aggregates stats from **Oct 1 through Feb 1** of each season (pre-All-Star cutoff).
3. Attaches All-Star labels from `allstar.csv`.
4. Saves the result to `data/historical_features.parquet`.

**Run once** – results are cached to disk, so subsequent runs are fast.

> **Prerequisites**: place `allstar.csv` in the `data/` folder (download from
> [Kaggle](https://www.kaggle.com/datasets/ethankeyes/nba-all-star-players-and-stats-1980-2022))
> before running this notebook.


In [ ]:
import sys, subprocess
packages = [
    "numpy", "pandas", "nba_api", "unidecode",
    "scikit-learn", "imbalanced-learn", "xgboost",
    "matplotlib", "joblib", "pyarrow", "tqdm",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Dependencies ready.")


In [ ]:
import sys
from pathlib import Path

# Make sure src/ is on the path when running from notebooks/
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
# Seasons for which we build historical features.
# The allstar.csv from Kaggle covers up to year=2022 (i.e. 2021-22 season).
# We include 2022-23 and 2023-24 whose labels are added manually below.
SEASONS = [
    "2008-09", "2009-10", "2010-11", "2011-12",
    "2012-13", "2013-14", "2014-15", "2015-16",
    "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22", "2022-23", "2023-24",
]

DATA_DIR   = REPO_ROOT / "data"
CACHE_DIR  = DATA_DIR / "cache"
ALLSTAR_CSV = DATA_DIR / "allstar.csv"
OUTPUT_PATH = DATA_DIR / "historical_features.parquet"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


## Additional All-Star labels (2022-23 and 2023-24)

The Kaggle CSV covers through year 2022 (= 2021-22 season).  We append the
two most recent complete seasons manually.  Player IDs were looked up via
`nba_api.stats.static.players`.


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

# 2022-23 All-Stars (year=2022 means the 2022-23 season)
# Player IDs are looked up by name in load_allstar_labels; we include them
# here only for reference and they will be resolved via name normalisation.
ALLSTARS_2022_23 = [
    # format: (first, last, year, team)
    # East
    ("Giannis",     "Antetokounmpo", 2022, "MIL"),
    ("Jayson",      "Tatum",          2022, "BOS"),
    ("Jaylen",      "Brown",          2022, "BOS"),
    ("Kevin",       "Durant",         2022, "PHO"),
    ("Kyrie",       "Irving",         2022, "DAL"),
    ("Julius",      "Randle",         2022, "NYK"),
    ("Bam",         "Adebayo",        2022, "MIA"),
    ("Donovan",     "Mitchell",       2022, "CLE"),
    ("Tyrese",      "Haliburton",     2022, "IND"),
    ("Jaren",       "Jackson",        2022, "MEM"),
    ("DeMar",       "DeRozan",        2022, "CHI"),
    # West
    ("Nikola",      "Jokic",          2022, "DEN"),
    ("LeBron",      "James",          2022, "LAL"),
    ("Luka",        "Doncic",         2022, "DAL"),
    ("Anthony",     "Davis",          2022, "LAL"),
    ("Shai",        "Gilgeous-Alexander", 2022, "OKC"),
    ("Zion",        "Williamson",     2022, "NOP"),
    ("De'Aaron",    "Fox",            2022, "SAC"),
    ("Domantas",    "Sabonis",        2022, "SAC"),
    ("Paul",        "George",         2022, "LAC"),
    ("Damian",      "Lillard",        2022, "POR"),
    ("Ja",          "Morant",         2022, "MEM"),
    ("Stephen",     "Curry",          2022, "GSW"),
    ("Paul",        "George",         2022, "LAC"),
]

# 2023-24 All-Stars (year=2023)
ALLSTARS_2023_24 = [
    # East
    ("Giannis",     "Antetokounmpo", 2023, "MIL"),
    ("Jayson",      "Tatum",          2023, "BOS"),
    ("Jaylen",      "Brown",          2023, "BOS"),
    ("Kevin",       "Durant",         2023, "PHO"),
    ("Bam",         "Adebayo",        2023, "MIA"),
    ("Tyrese",      "Haliburton",     2023, "IND"),
    ("Paolo",       "Banchero",       2023, "ORL"),
    ("Jalen",       "Brunson",        2023, "NYK"),
    ("Scottie",     "Barnes",         2023, "TOR"),
    ("Donovan",     "Mitchell",       2023, "CLE"),
    ("Damian",      "Lillard",        2023, "MIL"),
    # West
    ("Nikola",      "Jokic",          2023, "DEN"),
    ("LeBron",      "James",          2023, "LAL"),
    ("Luka",        "Doncic",         2023, "DAL"),
    ("Anthony",     "Davis",          2023, "LAL"),
    ("Shai",        "Gilgeous-Alexander", 2023, "OKC"),
    ("Zion",        "Williamson",     2023, "NOP"),
    ("De'Aaron",    "Fox",            2023, "SAC"),
    ("Karl-Anthony","Towns",          2023, "MIN"),
    ("Paul",        "George",         2023, "LAC"),
    ("Kawhi",       "Leonard",        2023, "LAC"),
    ("Stephen",     "Curry",          2023, "GSW"),
    ("Anthony",     "Edwards",        2023, "MIN"),
]

extra_cols = ["first", "last", "year", "team"]
extra_rows = ALLSTARS_2022_23 + ALLSTARS_2023_24
extra_df = pd.DataFrame(extra_rows, columns=extra_cols)
# Remove accidental duplicates within the extra data
extra_df = extra_df.drop_duplicates(subset=["first", "last", "year"])

if ALLSTAR_CSV.exists():
    base_df = pd.read_csv(ALLSTAR_CSV)
    # Only append rows for years not already present
    existing_years = set(base_df["year"].unique()) if "year" in base_df.columns else set()
    new_rows = extra_df[~extra_df["year"].isin(existing_years)]
    if not new_rows.empty:
        # Align columns – add any missing ones with NaN
        for col in base_df.columns:
            if col not in new_rows.columns:
                new_rows = new_rows.copy()
                new_rows[col] = np.nan
        shared_cols = [c for c in base_df.columns if c in new_rows.columns]
        base_df = pd.concat([base_df, new_rows[shared_cols]], ignore_index=True)
    base_df.to_csv(ALLSTAR_CSV, index=False)
    print(f"allstar.csv updated – {len(base_df)} rows total.")
else:
    extra_df.to_csv(ALLSTAR_CSV, index=False)
    print(f"allstar.csv created with {len(extra_df)} rows (recent seasons only).")
    print("NOTE: For full historical training data, download allstar.csv from Kaggle")
    print("  https://www.kaggle.com/datasets/ethankeyes/nba-all-star-players-and-stats-1980-2022")
    print("and place it in the data/ directory, then re-run this notebook.")


In [ ]:
from src.feature_engineering import build_historical_dataset

print("Building historical dataset…")
df = build_historical_dataset(
    seasons=SEASONS,
    allstar_csv=ALLSTAR_CSV,
    cache_dir=CACHE_DIR,
)

print(f"\nDataset shape: {df.shape}")
print(f"All-Star rate:  {df['allstar'].mean():.3%}")
print(f"Seasons:        {sorted(df['year'].unique())}")
df.head()


In [ ]:
df.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(df.dtypes)
